# Daily Credit Deterioration Signals\n\nBusiness logic:\n1. Compare current risk scores to a synthetic prior score baseline.\n2. Flag score drops and elevated PD changes.\n3. Persist daily deterioration signals for watchlist workflows.

In [ ]:
from pyspark.sql import functions as F\n\nrisk_scores = spark.table(\"Risk_CreditRiskScores\")\n\n# Build a prior baseline for demo purposes. In production, replace with prior-day snapshot table.\nbaseline = (\n    risk_scores\n    .withColumn(\"PrevScore\", F.col(\"Score\") + (F.rand(101) * 55 - 15).cast(\"int\"))\n    .withColumn(\"PrevPD\", F.round(F.col(\"PD\") + (F.rand(102) * 0.015 - 0.004), 4))\n    .select(\"CustomerID\", \"PrevScore\", \"PrevPD\")\n)\n\ndeterioration = (\n    risk_scores.alias(\"c\")\n    .join(baseline.alias(\"p\"), on=\"CustomerID\", how=\"left\")\n    .withColumn(\"ScoreDrop\", F.col(\"p.PrevScore\") - F.col(\"c.Score\"))\n    .withColumn(\"PDDelta\", F.round(F.col(\"c.PD\") - F.col(\"p.PrevPD\"), 4))\n    .withColumn(\"DeteriorationFlag\", (F.col(\"ScoreDrop\") >= 30) | (F.col(\"PDDelta\") >= 0.01))\n    .withColumn(\"Severity\", F.when(F.col(\"ScoreDrop\") >= 60, F.lit(\"High\")).when(F.col(\"ScoreDrop\") >= 30, F.lit(\"Medium\")).otherwise(F.lit(\"Low\")))\n    .withColumn(\"AsOfDate\", F.current_date())\n    .select(\"AsOfDate\", \"CustomerID\", \"Score\", \"PrevScore\", \"ScoreDrop\", \"PD\", \"PrevPD\", \"PDDelta\", \"DeteriorationFlag\", \"Severity\")\n)\n\n(\n    deterioration\n    .write.format(\"delta\")\n    .mode(\"overwrite\")\n    .option(\"overwriteSchema\", \"true\")\n    .saveAsTable(\"Risk_DailyCreditDeterioration\")\n)\n\ndisplay(deterioration.filter(F.col(\"DeteriorationFlag\") == True).orderBy(F.desc(\"ScoreDrop\")).limit(50))\n